In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in {'Baseline_Training', 'CTC_models', 'PhoneticFeatures_Training'}:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from project_paths import (
    BASELINE_MODELS_DIR,
    CTC_MODELS_DIR,
    DEFAULT_BASELINE_MODEL,
    DEFAULT_CTC_ATTENTION_MODEL,
    DEFAULT_CTC_MODEL,
    DEFAULT_PHONETIC_MODEL_2CLASS,
    DEFAULT_PHONETIC_MODEL_3CLASS,
    EXAMPLE_EMBEDDINGS,
    EXAMPLE_PHONEMES,
    EXAMPLE_SEG_B2,
    EXAMPLE_SEG_B4,
    EXAMPLE_WAV,
    PHONEME_CHOICES_SUMMARY,
    PHONETIC_MODELS_DIR,
    TABLES_DIR,
)


In [ ]:
import random
import os
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

from GetData import load_selected_embeddings
from ToIPA import corpres_to_ipa_symbol, panphon_features
from phonetic_feature_utils import (
    FEATURE_NAMES,
    PhonemeEmbeddingDataset,
    PhonemeFeaturePredictor,
    build_prediction_rows,
    collect_symbol_prediction_stats,
    compute_class_weights,
    compute_confusion_matrices,
    evaluate_before_training,
    infer_num_classes,
    inspect_single_prediction,
    plot_confusion_matrices,
    plot_confusion_matrices_norm,
    plot_feature_accuracy,
    plot_symbol_prediction_distributions,
    plot_training,
    train_model,
    compute_balanced_accuracy
)



In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA devices: {torch.cuda.device_count()}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA runtime version: {torch.version.cuda}")
print(f"cuDNN version: {torch.backends.cudnn.version() if torch.cuda.is_available() else 'N/A'}")

# Проверь установку PyTorch
print(f"torch.cuda.is_built(): {torch.backends.cudnn.enabled}")



In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

## Cleaned copy
Эта копия вычищает нестабильные части пайплайна: обучение, confusion matrices и блок пост-анализа предсказаний.


In [ ]:
root_dir = r'\\nid-tts-02\mnt\hot_store\trainee_common\ananeva\CORPRES_embs_no_averaging'
save_path = str(PROJECT_ROOT)
speakers = ['Vladimir', 'Maria', 'Mikhail', 'Anna', 'Galina', 'Victoria', 'Petr', 'Alexander']
phonetic_features = ['syl', 'son', 'cons', 'cont', 'delrel', 'lat', 'nas', 'strid', 'voi', 'sg', 'cg', 'ant', 'cor', 'distr', 'lab', 'hi', 'lo', 'back', 'round', 'velaric', 'tense', 'long']
phoneme_list_full = ['a0', 'a1', 'a2', 'a4', 'b', "b'", 'c', 'ch', 'ch_', 'd', "d'", 'e0', 'e1', 'e2', 'e4', 'f', "f'", 'g', "g'", 'h', "h'", 'i0', 'i1', 'i2', 'i4', 'j', 'jr', 'jl', 'ji4', 'k', "k'", 'l', "l'", 'm', "m'", 'n', "n'", 'o0', 'o1', 'o2', 'o4', 'p', "p'", 'r', "r'", 's', "s'", 'sc', 'sh', 't', "t'", 'u0', 'u1', 'u2', 'u4', 'v', "v'", 'y0', 'y1', 'y2', 'y4', 'z', "z'", 'zh', "zh'", 'C', 'CH', 'H', 'SC']

In [ ]:
debugging = False
save_exp = False
training = True
#model_id = 'VACXUXVEXO'
training_mode = 'triplet' # ['triplet', 'ctc', 'averaged']
phoneme_list = ['a0', 'a4', 'b', "b'", 'c', 'ch', 'd', "d'", 'e0', 'f', "f'", 'g', 'h', 'i0','i4', 'j', 'k', "k'", 'l', "l'", 'm', "m'", 'n', "n'", 'o0', 'p', "p'", 'r', "r'", 's', "s'", 'sh', 't', "t'", 'u0', 'v', "v'", 'y0', 'z', "z'", 'zh', 'a1', 'i1','u1', 'y1', ]
# phoneme_list = ['a0', 'a4', 'b', 'd','e0', 'f',  'g', 'h', 'i0','i4', 'j', 'k','l', 'm',  'n', 'o0', 'p', 'r', 's', 'sh', 't', 'u0', 'v',  'y0', 'z', 'zh', 'sil']
# phoneme_list = ['a0', 't', "i0", "t'", "k", 's', "s'" ]  
# интересующие фонемы (без позиций)
samples_of_ph = 1000              # сколько примеров на фонему
train_speakers = ['Vladimir', 'Maria', 'Mikhail', 'Anna']
test_speakers = ['Galina', 'Victoria', 'Petr', 'Alexander']

if debugging:
    phoneme_list = ['a0', 't', "i0" ]  
    samples_of_ph = 1            
    train_speakers = ['Vladimir']
    test_speakers = ['Galina']
    

In [ ]:
int_code = [random.randint(65, 90) for i in range(10)]
uid = ''.join([chr(code) for code in int_code])

In [ ]:
log_path = save_path + '\\runs\\' + uid

In [ ]:
num_classes = len(phoneme_list)

In [ ]:
learning_rate = 1e-3
weight_decay=1e-4
batch_size = 32
num_epochs = 20
dropout = 0.1
input_size = 512

In [ ]:
save_model_path = r'C:\Users\ext-ananeva\phon_clust\gitlab\ssl_phoneme_clusterizer\Main_project\phonetic_features_predictions'

In [ ]:
# train_embeddings, train_phonemes_nobord, _ = load_selected_embeddings(root_dir, phoneme_list, samples_of_ph, train_speakers,  triplet_variation='no borders')
# 
# test_embeddings, test_phonemes_nobord, _ = load_selected_embeddings(root_dir, phoneme_list, samples_of_ph, test_speakers,  triplet_variation='no borders')

train_embeddings = torch.load("train_embeddings.pt")
test_embeddings = torch.load("test_embeddings.pt")


In [ ]:
# moved to phonetic_feature_utils.py


In [ ]:
# moved to phonetic_feature_utils.py


In [ ]:
train_dataset = PhonemeEmbeddingDataset(train_embeddings, phoneme_list, corpres_to_ipa_symbol=corpres_to_ipa_symbol, panphon_features=panphon_features)
test_dataset = PhonemeEmbeddingDataset(test_embeddings, phoneme_list, corpres_to_ipa_symbol=corpres_to_ipa_symbol, panphon_features=panphon_features)

In [ ]:
train_dataset[-1]

In [ ]:
label_counter = Counter()
phoneme_counter = Counter()
ipa_counter = Counter()
feature_sum = None
feature_count = 0

for i in range(len(train_dataset)):
    _, labels, phonemes, ipa_symbols, features = train_dataset[i]

    # если не триплет - приводим всё к списку
    if not train_dataset.triplet_labels:
        labels = [labels.item()]
        phonemes = [phonemes[0]]
        ipa_symbols = [ipa_symbols[0]]
        features = [features] if features is not None else []

    # считаем частоты
    for l in labels:
        label_counter[int(l)] += 1
    for ph in phonemes:
        phoneme_counter[ph] += 1
    for ipa in ipa_symbols:
        ipa_counter[ipa] += 1        

In [ ]:
plt.figure()
plt.bar(phoneme_counter.keys(), phoneme_counter.values())
plt.xticks(rotation=90)
plt.title("Phoneme Distribution")
plt.show()

In [ ]:
plt.figure()
plt.bar(ipa_counter.keys(), ipa_counter.values())
plt.xticks(rotation=90)
plt.title("IPA Distribution")
plt.show()

In [ ]:
# будем считать: axis=0 — признаки, axis=1 — значение (0,1,2)
feature_counts = None

for i in range(len(train_dataset)):
    _, labels, phonemes, ipa_symbols, features = train_dataset[i]

    if not train_dataset.triplet_labels:
        labels = [labels.item()]
        phonemes = [phonemes[0]]
        ipa_symbols = [ipa_symbols[0]]
        features = [features] if features is not None else []

    for f in features:
        f = f.numpy().astype(int)  # предполагаем значения 0/1/2, форма (num_features,)

        if feature_counts is None:
            # shape: (num_features, 3) — для значений 0,1,2
            feature_counts = np.zeros((len(f), 3), dtype=int)

        # проверим, что все значения в {0,1,2}
        assert ((f >= 0) & (f <= 2)).all()

        for val in (0, 1, 2):
            feature_counts[:, val] += (f == val).astype(int)

# построение сгруппированной гистограммы
if feature_counts is not None:
    num_features = feature_counts.shape[0]

    # увеличим расстояние между признаками
    group_spacing = 1.5  # чем больше, тем больше отступ между признаками
    x = np.arange(num_features) * group_spacing

    width = 0.25  # ширина одного столбца

    plt.figure(figsize=(14, 6))
    plt.bar(x - width, feature_counts[:, 0], width=width, label='0')
    plt.bar(x,          feature_counts[:, 1], width=width, label='1')
    plt.bar(x + width, feature_counts[:, 2], width=width, label='2')

    plt.xticks(x, phonetic_features, rotation=90)
    plt.ylabel("Count")
    plt.title("Feature value counts (0/1/2) per feature")
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
input_size = len(train_dataset[0][0])
num_features = len(train_dataset[0][-1])
train_dataset[0][-1]


In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [ ]:
# moved to phonetic_feature_utils.py


In [ ]:
# moved to phonetic_feature_utils.py


In [ ]:
compute_class_weights(train_dataset, num_features)

In [ ]:
NUM_FEATURES = len(FEATURE_NAMES)
FEATURE_NAMES



In [ ]:
# moved to phonetic_feature_utils.py


In [ ]:
infer_num_classes(train_dataset)

In [ ]:
# moved to phonetic_feature_utils.py


In [ ]:
# moved to phonetic_feature_utils.py


In [ ]:
# moved to phonetic_feature_utils.py


In [ ]:
# moved to phonetic_feature_utils.py


In [ ]:
# moved to phonetic_feature_utils.py


In [ ]:
# moved to phonetic_feature_utils.py


In [ ]:
def plot_feature_accuracy(history, baseline_feature_acc, feature_names, active_heads):
    steps = history["steps"]

    for head_idx in active_heads:
        if head_idx not in baseline_feature_acc:
            continue

        plt.figure()
        plt.scatter(0, baseline_feature_acc[head_idx], label="baseline")
        plt.plot(steps, history["train_feature_acc"][head_idx], label="train")
        plt.plot(steps, history["val_feature_acc"][head_idx], label="val")
        plt.title(f"Feature '{feature_names[head_idx]}' accuracy")
        plt.xlabel("step")
        plt.ylabel("accuracy")
        plt.legend()
        plt.show()


import matplotlib.pyplot as plt


def plot_training(history, baseline_loss, baseline_acc):
    steps = history["steps"]

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.scatter(0, baseline_loss, label="baseline")
    plt.plot(steps, history["train_loss"], label="train")
    plt.plot(steps, history["val_loss"], label="val")
    plt.title("Loss")
    plt.xlabel("step")
    plt.ylabel("loss")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.scatter(0, baseline_acc, label="baseline")
    plt.plot(steps, history["train_acc"], label="train")
    plt.plot(steps, history["val_acc"], label="val")
    plt.title("Accuracy")
    plt.xlabel("step")
    plt.ylabel("accuracy")
    plt.legend()

    plt.tight_layout()
    plt.show()



In [ ]:
num_classes_per_feature = infer_num_classes(train_dataset)

In [ ]:
#num_classes_per_feature = [3] * NUM_FEATURES

In [ ]:
#active_heads = list(range(NUM_FEATURES))
active_heads = [FEATURE_NAMES.index(f) for f in ['syl', 'son', 'cons', 'cont', 'delrel', 'lat', 'nas', 'strid',
                                                'voi', 'ant', 'cor', 'distr', 'lab',
                                                'hi', 'lo', 'back', 'round',]]

active_heads_names = ['syl', 'son', 'cons', 'cont', 'delrel', 'lat', 'nas', 'strid',
                                                'voi', 'ant', 'cor', 'distr', 'lab',
                                                'hi', 'lo', 'back', 'round',
                                                'tense']
                                                

In [ ]:
model = PhonemeFeaturePredictor(
    input_dim=256,
    hidden_dim=256,
    num_classes_per_feature=num_classes_per_feature,
    active_heads=active_heads
).to(device)
criterion_dict = {}

In [ ]:
for i in active_heads:
    criterion_dict[i] = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=10,
    gamma=0.5
)

In [ ]:
baseline_loss, baseline_acc, baseline_feature_acc = evaluate_before_training(
    model,
    test_loader,
    device,
    active_heads,
    criterion_dict
)

In [ ]:
# history = {
#     "train_loss": [], "test_loss": [],
#     "train_acc": [], "test_acc": [],
#     "train_feature_acc": {i: [] for i in active_heads},
#     "test_feature_acc": {i: [] for i in active_heads},
#     # новые поля для половин
#     "train_half_loss": [], "test_half_loss": [],
#     "train_half_acc": [], "test_half_acc": [],
#     "train_half_feature_acc": {i: [] for i in active_heads},
#     "test_half_feature_acc": {i: [] for i in active_heads},
# }

In [ ]:

history = train_model(
    model,
    train_loader,
    test_loader,
    optimizer,
    scheduler,
    device,
    active_heads,
    criterion_dict,
    epochs=3,
    eval_steps_per_epoch=19,
    save_model_path=save_model_path
)



In [ ]:
plot_training(history, baseline_loss, baseline_acc)
plot_feature_accuracy(history, baseline_feature_acc, FEATURE_NAMES, active_heads)



In [ ]:
# moved to phonetic_feature_utils.py


In [ ]:
# moved to phonetic_feature_utils.py


In [ ]:
# moved to phonetic_feature_utils.py


In [ ]:
confusion_matrices = compute_confusion_matrices(model, test_loader, device, active_heads)


In [ ]:
plot_confusion_matrices(confusion_matrices, FEATURE_NAMES)

In [ ]:
plot_confusion_matrices_norm(confusion_matrices, FEATURE_NAMES)

## Inference

In [ ]:
checkpoint = torch.load(save_model_path, map_location=device)

model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
if checkpoint["scheduler_state_dict"] is not None:
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

In [ ]:
model.eval()


In [ ]:
inspect_single_prediction(model, test_dataset, device, active_heads, FEATURE_NAMES, skip_symbols={"k", "a"})


In [ ]:
stats, true_classes = collect_symbol_prediction_stats(model, test_dataset, device, active_heads)


In [ ]:
symbol_features = [
    {
        "symbol": symbol,
        "head": head_id,
        "counts": dict(class_counts),
        "true_class": true_classes[(symbol, head_id)],
    }
    for symbol, heads_dict in stats.items()
    for head_id, class_counts in heads_dict.items()
]


In [ ]:
symbol_features


In [ ]:
true_classes


In [ ]:
prediction_rows = build_prediction_rows(stats, true_classes)


In [ ]:
prediction_rows[:5]


In [ ]:
import pandas as pd

plot_symbol_prediction_distributions(prediction_rows, FEATURE_NAMES)


In [ ]:
pd.DataFrame(prediction_rows).head()
